In [1]:
# =============================================================================
# PROJECT       : Banking Stress Testing Framework
# VERSION       : Python V2 Migration
# MODULE        : 03 - Risk Segmentation
# NOTEBOOK      : 03_Risk_Segmentation.ipynb
# AUTHOR        : Umesh Zure
#
# PURPOSE:
# Assign SAS V1-equivalent Credit Risk segments and DPD delinquency buckets
# to the prepared banking portfolio.
#
# INPUT:
# output/prepared_portfolio.csv
#
# OUTPUTS:
# output/segmented_portfolio.csv
# validation/03_risk_segmentation_summary.csv
# validation/03_risk_segmentation_validation.csv
# validation/03_risk_segment_distribution.csv
# validation/03_dpd_bucket_distribution.csv
# validation/03_risk_dpd_cross_tab.csv
# logs/03_Risk_Segmentation.log
#
# SAS V1 BUSINESS RULES:
#
# Risk Segment:
#   Missing Credit_Score  -> Unclassified
#   Credit_Score >= 750   -> Low Risk
#   Credit_Score >= 650   -> Medium Risk
#   Credit_Score < 650    -> High Risk
#
# DPD Bucket:
#   Missing Current_DPD   -> Unresolved
#   Current_DPD = 0       -> Current
#   Current_DPD 1-30      -> 1-30
#   Current_DPD 31-60     -> 31-60
#   Current_DPD 61-90     -> 61-90
#   Current_DPD > 90      -> 90+
#   Other values          -> Invalid DPD
# =============================================================================

from datetime import datetime

module_start_timestamp = datetime.now()

print("=" * 80)
print("BANKING STRESS TESTING FRAMEWORK")
print("PYTHON V2 MIGRATION")
print("MODULE : 03_RISK_SEGMENTATION")
print(
    "EXECUTION STARTED :",
    module_start_timestamp.strftime(
        "%Y-%m-%d %H:%M:%S"
    )
)
print("=" * 80)

BANKING STRESS TESTING FRAMEWORK
PYTHON V2 MIGRATION
MODULE : 03_RISK_SEGMENTATION
EXECUTION STARTED : 2026-10-13 09:27:39


In [2]:
# =============================================================================
# BLOCK 1: ENVIRONMENT, PATHS, AND LOGGING
# =============================================================================

block_start_time = datetime.now()
block_status = "STARTED"

print("=" * 80)
print("MODULE 03: RISK SEGMENTATION")
print("BLOCK 1 : ENVIRONMENT, PATHS, AND LOGGING")
print(
    "START TIME :",
    block_start_time.strftime(
        "%Y-%m-%d %H:%M:%S"
    )
)
print("=" * 80)

try:
    from google.colab import drive
    from pathlib import Path
    import logging
    import sys

    import numpy as np
    import pandas as pd

    drive.mount(
        "/content/drive"
    )

    PROJECT_ROOT = Path(
        "/content/drive/MyDrive/"
        "Banking_Stress_Testing_V2_Python"
    )

    INPUT_FILE = (
        PROJECT_ROOT
        / "output"
        / "prepared_portfolio.csv"
    )

    OUTPUT_FILE = (
        PROJECT_ROOT
        / "output"
        / "segmented_portfolio.csv"
    )

    VALIDATION_DIR = (
        PROJECT_ROOT
        / "validation"
    )

    LOG_DIR = (
        PROJECT_ROOT
        / "logs"
    )

    SUMMARY_FILE = (
        VALIDATION_DIR
        / "03_risk_segmentation_summary.csv"
    )

    VALIDATION_FILE = (
        VALIDATION_DIR
        / "03_risk_segmentation_validation.csv"
    )

    RISK_DISTRIBUTION_FILE = (
        VALIDATION_DIR
        / "03_risk_segment_distribution.csv"
    )

    DPD_DISTRIBUTION_FILE = (
        VALIDATION_DIR
        / "03_dpd_bucket_distribution.csv"
    )

    CROSS_TAB_FILE = (
        VALIDATION_DIR
        / "03_risk_dpd_cross_tab.csv"
    )

    LOG_FILE = (
        LOG_DIR
        / "03_Risk_Segmentation.log"
    )

    OUTPUT_FILE.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    VALIDATION_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    LOG_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    logger = logging.getLogger(
        "03_Risk_Segmentation"
    )

    logger.setLevel(
        logging.INFO
    )

    if logger.hasHandlers():
        logger.handlers.clear()

    log_formatter = logging.Formatter(
        "%(asctime)s | %(levelname)s | %(message)s"
    )

    file_handler = logging.FileHandler(
        LOG_FILE,
        mode="w",
        encoding="utf-8"
    )

    file_handler.setFormatter(
        log_formatter
    )

    console_handler = logging.StreamHandler(
        sys.stdout
    )

    console_handler.setFormatter(
        log_formatter
    )

    logger.addHandler(
        file_handler
    )

    logger.addHandler(
        console_handler
    )

    assert PROJECT_ROOT.exists(), (
        f"Project root does not exist: {PROJECT_ROOT}"
    )

    assert INPUT_FILE.exists(), (
        f"Module 02 output does not exist: {INPUT_FILE}"
    )

    logger.info(
        "Environment, paths, and logging configured successfully."
    )

    print("Project Root Exists:", PROJECT_ROOT.exists())
    print("Module 02 Output Exists:", INPUT_FILE.exists())
    print("Input File:", INPUT_FILE)
    print("Segmented Output:", OUTPUT_FILE)
    print("Validation Directory:", VALIDATION_DIR)
    print("Execution Log:", LOG_FILE)

    block_status = "SUCCESS"

except Exception:
    block_status = "FAILED"

    if "logger" in locals():
        logger.exception(
            "Block 1 failed during environment configuration."
        )

    raise

finally:
    block_end_time = datetime.now()

    block_duration_seconds = round(
        (
            block_end_time
            - block_start_time
        ).total_seconds(),
        3
    )

    print("=" * 80)
    print("BLOCK STATUS :", block_status)
    print(
        "END TIME     :",
        block_end_time.strftime(
            "%Y-%m-%d %H:%M:%S"
        )
    )
    print(
        "DURATION (SECONDS):",
        block_duration_seconds
    )
    print("=" * 80)

MODULE 03: RISK SEGMENTATION
BLOCK 1 : ENVIRONMENT, PATHS, AND LOGGING
START TIME : 2026-10-13 09:30:29
Mounted at /content/drive
2026-10-13 09:30:56,915 | INFO | Environment, paths, and logging configured successfully.


INFO:03_Risk_Segmentation:Environment, paths, and logging configured successfully.


Project Root Exists: True
Module 02 Output Exists: True
Input File: /content/drive/MyDrive/Banking_Stress_Testing_V2_Python/output/prepared_portfolio.csv
Segmented Output: /content/drive/MyDrive/Banking_Stress_Testing_V2_Python/output/segmented_portfolio.csv
Validation Directory: /content/drive/MyDrive/Banking_Stress_Testing_V2_Python/validation
Execution Log: /content/drive/MyDrive/Banking_Stress_Testing_V2_Python/logs/03_Risk_Segmentation.log
BLOCK STATUS : SUCCESS
END TIME     : 2026-10-13 09:30:56
DURATION (SECONDS): 27.595


In [3]:
# =============================================================================
# BLOCK 2: LOAD AND VALIDATE MODULE 02 OUTPUT
# =============================================================================

block_start_time = datetime.now()
block_status = "STARTED"

print("=" * 80)
print("MODULE 03: RISK SEGMENTATION")
print("BLOCK 2 : LOAD AND VALIDATE MODULE 02 OUTPUT")
print(
    "START TIME :",
    block_start_time.strftime(
        "%Y-%m-%d %H:%M:%S"
    )
)
print("=" * 80)

try:
    REQUIRED_COLUMNS = [
        "Customer_ID",
        "Loan_ID",
        "Credit_Score",
        "Current_DPD",
        "Customer_Segment",
        "Product_Type",
        "Outstanding_Balance",
        "Reporting_Date",
    ]

    prepared_portfolio = pd.read_csv(
        INPUT_FILE,
        low_memory=False
    )

    segmented_portfolio = prepared_portfolio.copy(
        deep=True
    )

    input_record_count = len(
        segmented_portfolio
    )

    input_column_count = len(
        segmented_portfolio.columns
    )

    missing_required_columns = sorted(
        set(REQUIRED_COLUMNS)
        - set(segmented_portfolio.columns)
    )

    if missing_required_columns:
        raise ValueError(
            "Required columns are missing: "
            f"{missing_required_columns}"
        )

    segmented_portfolio[
        "Credit_Score"
    ] = pd.to_numeric(
        segmented_portfolio[
            "Credit_Score"
        ],
        errors="coerce"
    )

    segmented_portfolio[
        "Current_DPD"
    ] = pd.to_numeric(
        segmented_portfolio[
            "Current_DPD"
        ],
        errors="coerce"
    )

    duplicate_customer_ids_before = int(
        segmented_portfolio[
            "Customer_ID"
        ]
        .duplicated()
        .sum()
    )

    duplicate_loan_ids_before = int(
        segmented_portfolio[
            "Loan_ID"
        ]
        .duplicated()
        .sum()
    )

    missing_credit_scores_before = int(
        segmented_portfolio[
            "Credit_Score"
        ]
        .isna()
        .sum()
    )

    missing_current_dpd_before = int(
        segmented_portfolio[
            "Current_DPD"
        ]
        .isna()
        .sum()
    )

    print("=" * 80)
    print("MODULE 02 OUTPUT LOADED SUCCESSFULLY")
    print("=" * 80)

    print("Input Records:", input_record_count)
    print("Input Columns:", input_column_count)
    print(
        "Missing Required Columns:",
        missing_required_columns
    )
    print(
        "Missing Credit Scores:",
        missing_credit_scores_before
    )
    print(
        "Missing Current DPD:",
        missing_current_dpd_before
    )
    print(
        "Duplicate Customer_ID:",
        duplicate_customer_ids_before
    )
    print(
        "Duplicate Loan_ID:",
        duplicate_loan_ids_before
    )
    print("Input Validation Status: PASS")

    display(
        segmented_portfolio[
            [
                "Customer_ID",
                "Loan_ID",
                "Credit_Score",
                "Current_DPD",
                "Customer_Segment",
                "Product_Type",
            ]
        ]
        .head(10)
    )

    logger.info(
        "Module 02 output loaded successfully: "
        "%s rows and %s columns.",
        input_record_count,
        input_column_count
    )

    block_status = "SUCCESS"

except Exception:
    block_status = "FAILED"

    logger.exception(
        "Block 2 failed while loading Module 02 output."
    )

    raise

finally:
    block_end_time = datetime.now()

    block_duration_seconds = round(
        (
            block_end_time
            - block_start_time
        ).total_seconds(),
        3
    )

    print("=" * 80)
    print("BLOCK STATUS :", block_status)
    print(
        "END TIME     :",
        block_end_time.strftime(
            "%Y-%m-%d %H:%M:%S"
        )
    )
    print(
        "DURATION (SECONDS):",
        block_duration_seconds
    )
    print("=" * 80)


MODULE 03: RISK SEGMENTATION
BLOCK 2 : LOAD AND VALIDATE MODULE 02 OUTPUT
START TIME : 2026-10-13 09:33:09
MODULE 02 OUTPUT LOADED SUCCESSFULLY
Input Records: 10000
Input Columns: 22
Missing Required Columns: []
Missing Credit Scores: 0
Missing Current DPD: 0
Duplicate Customer_ID: 0
Duplicate Loan_ID: 0
Input Validation Status: PASS


,Customer_ID,Loan_ID,Credit_Score,Current_DPD,Customer_Segment,Product_Type
0,1000001,50000001,756.0,0.0,Retail,Personal Loan
1,1000002,50000002,817.0,0.0,Corporate,Corporate Loan
2,1000003,50000003,630.0,0.0,Retail,Credit Card
3,1000004,50000004,724.0,0.0,SME,Auto Loan
4,1000005,50000005,784.0,0.0,Retail,Personal Loan
5,1000006,50000006,782.0,0.0,Retail,Credit Card
6,1000007,50000007,881.0,0.0,Retail,Auto Loan
7,1000008,50000008,710.0,0.0,SME,SME Loan
8,1000009,50000009,653.0,0.0,Retail,Credit Card
9,1000010,50000010,732.0,0.0,Retail,Mortgage


2026-10-13 09:33:09,559 | INFO | Module 02 output loaded successfully: 10000 rows and 22 columns.


INFO:03_Risk_Segmentation:Module 02 output loaded successfully: 10000 rows and 22 columns.


BLOCK STATUS : SUCCESS
END TIME     : 2026-10-13 09:33:09
DURATION (SECONDS): 0.345


In [4]:
# =============================================================================
# BLOCK 3: ASSIGN CREDIT RISK SEGMENT
# =============================================================================

block_start_time = datetime.now()
block_status = "STARTED"

print("=" * 80)
print("MODULE 03: RISK SEGMENTATION")
print("BLOCK 3 : ASSIGN CREDIT RISK SEGMENT")
print(
    "START TIME :",
    block_start_time.strftime(
        "%Y-%m-%d %H:%M:%S"
    )
)
print("=" * 80)

try:
    risk_conditions = [
        segmented_portfolio[
            "Credit_Score"
        ].isna(),

        segmented_portfolio[
            "Credit_Score"
        ] >= 750,

        segmented_portfolio[
            "Credit_Score"
        ] >= 650,

        segmented_portfolio[
            "Credit_Score"
        ] < 650,
    ]

    risk_values = [
        "Unclassified",
        "Low Risk",
        "Medium Risk",
        "High Risk",
    ]

    segmented_portfolio[
        "Risk_Segment"
    ] = np.select(
        risk_conditions,
        risk_values,
        default="Unclassified"
    )

    allowed_risk_segments = [
        "Low Risk",
        "Medium Risk",
        "High Risk",
        "Unclassified",
    ]

    invalid_risk_segment_count = int(
        (
            ~segmented_portfolio[
                "Risk_Segment"
            ].isin(
                allowed_risk_segments
            )
        )
        .sum()
    )

    missing_risk_segment_count = int(
        segmented_portfolio[
            "Risk_Segment"
        ]
        .isna()
        .sum()
    )

    risk_segment_distribution = (
        segmented_portfolio[
            "Risk_Segment"
        ]
        .value_counts(
            dropna=False
        )
        .reindex(
            allowed_risk_segments,
            fill_value=0
        )
        .rename_axis(
            "Risk_Segment"
        )
        .reset_index(
            name="Record_Count"
        )
    )

    risk_segment_distribution[
        "Portfolio_Percentage"
    ] = (
        risk_segment_distribution[
            "Record_Count"
        ]
        / input_record_count
        * 100
    ).round(2)

    risk_segment_total = int(
        risk_segment_distribution[
            "Record_Count"
        ].sum()
    )

    assert risk_segment_total == input_record_count, (
        "Risk Segment distribution does not reconcile "
        "to the input record count."
    )

    assert invalid_risk_segment_count == 0, (
        "Unexpected Risk Segment values were created."
    )

    assert missing_risk_segment_count == 0, (
        "Missing Risk Segment values were created."
    )

    print("=" * 80)
    print("CREDIT RISK SEGMENT ASSIGNMENT COMPLETED")
    print("=" * 80)

    print(
        "Segmented Records:",
        risk_segment_total
    )
    print(
        "Invalid Risk Segments:",
        invalid_risk_segment_count
    )
    print(
        "Missing Risk Segments:",
        missing_risk_segment_count
    )
    print(
        "Risk Segment Reconciliation: PASS"
    )

    display(
        risk_segment_distribution
    )

    print("\nSample Segmented Records:")

    display(
        segmented_portfolio[
            [
                "Customer_ID",
                "Loan_ID",
                "Credit_Score",
                "Risk_Segment",
            ]
        ]
        .head(15)
    )

    logger.info(
        "Credit Risk Segment assignment completed "
        "for %s records.",
        risk_segment_total
    )

    block_status = "SUCCESS"

except Exception:
    block_status = "FAILED"

    logger.exception(
        "Block 3 failed during Risk Segment assignment."
    )

    raise

finally:
    block_end_time = datetime.now()

    block_duration_seconds = round(
        (
            block_end_time
            - block_start_time
        ).total_seconds(),
        3
    )

    print("=" * 80)
    print("BLOCK STATUS :", block_status)
    print(
        "END TIME     :",
        block_end_time.strftime(
            "%Y-%m-%d %H:%M:%S"
        )
    )
    print(
        "DURATION (SECONDS):",
        block_duration_seconds
    )
    print("=" * 80)

MODULE 03: RISK SEGMENTATION
BLOCK 3 : ASSIGN CREDIT RISK SEGMENT
START TIME : 2026-10-13 09:35:21
CREDIT RISK SEGMENT ASSIGNMENT COMPLETED
Segmented Records: 10000
Invalid Risk Segments: 0
Missing Risk Segments: 0
Risk Segment Reconciliation: PASS


,Risk_Segment,Record_Count,Portfolio_Percentage
0,Low Risk,2535,25.35
1,Medium Risk,4980,49.80
2,High Risk,2485,24.85
3,Unclassified,0,0.00



Sample Segmented Records:


,Customer_ID,Loan_ID,Credit_Score,Risk_Segment
0,1000001,50000001,756.0,Low Risk
1,1000002,50000002,817.0,Low Risk
2,1000003,50000003,630.0,High Risk
3,1000004,50000004,724.0,Medium Risk
4,1000005,50000005,784.0,Low Risk
5,1000006,50000006,782.0,Low Risk
6,1000007,50000007,881.0,Low Risk
7,1000008,50000008,710.0,Medium Risk
8,1000009,50000009,653.0,Medium Risk
9,1000010,50000010,732.0,Medium Risk


2026-10-13 09:35:21,654 | INFO | Credit Risk Segment assignment completed for 10000 records.


INFO:03_Risk_Segmentation:Credit Risk Segment assignment completed for 10000 records.


BLOCK STATUS : SUCCESS
END TIME     : 2026-10-13 09:35:21
DURATION (SECONDS): 0.04


In [5]:
# =============================================================================
# BLOCK 4: ASSIGN DPD DELINQUENCY BUCKET
# =============================================================================

block_start_time = datetime.now()
block_status = "STARTED"

print("=" * 80)
print("MODULE 03: RISK SEGMENTATION")
print("BLOCK 4 : ASSIGN DPD DELINQUENCY BUCKET")
print(
    "START TIME :",
    block_start_time.strftime(
        "%Y-%m-%d %H:%M:%S"
    )
)
print("=" * 80)

try:
    dpd_conditions = [
        segmented_portfolio[
            "Current_DPD"
        ].isna(),

        segmented_portfolio[
            "Current_DPD"
        ] == 0,

        segmented_portfolio[
            "Current_DPD"
        ].between(
            1,
            30,
            inclusive="both"
        ),

        segmented_portfolio[
            "Current_DPD"
        ].between(
            31,
            60,
            inclusive="both"
        ),

        segmented_portfolio[
            "Current_DPD"
        ].between(
            61,
            90,
            inclusive="both"
        ),

        segmented_portfolio[
            "Current_DPD"
        ] > 90,
    ]

    dpd_values = [
        "Unresolved",
        "Current",
        "1-30",
        "31-60",
        "61-90",
        "90+",
    ]

    segmented_portfolio[
        "DPD_Bucket"
    ] = np.select(
        dpd_conditions,
        dpd_values,
        default="Invalid DPD"
    )

    allowed_dpd_buckets = [
        "Current",
        "1-30",
        "31-60",
        "61-90",
        "90+",
        "Unresolved",
        "Invalid DPD",
    ]

    invalid_dpd_bucket_values = int(
        (
            ~segmented_portfolio[
                "DPD_Bucket"
            ].isin(
                allowed_dpd_buckets
            )
        )
        .sum()
    )

    missing_dpd_bucket_count = int(
        segmented_portfolio[
            "DPD_Bucket"
        ]
        .isna()
        .sum()
    )

    dpd_bucket_distribution = (
        segmented_portfolio[
            "DPD_Bucket"
        ]
        .value_counts(
            dropna=False
        )
        .reindex(
            allowed_dpd_buckets,
            fill_value=0
        )
        .rename_axis(
            "DPD_Bucket"
        )
        .reset_index(
            name="Record_Count"
        )
    )

    dpd_bucket_distribution[
        "Portfolio_Percentage"
    ] = (
        dpd_bucket_distribution[
            "Record_Count"
        ]
        / input_record_count
        * 100
    ).round(2)

    dpd_bucket_total = int(
        dpd_bucket_distribution[
            "Record_Count"
        ].sum()
    )

    assert dpd_bucket_total == input_record_count, (
        "DPD Bucket distribution does not reconcile "
        "to the input record count."
    )

    assert invalid_dpd_bucket_values == 0, (
        "Unexpected DPD Bucket values were created."
    )

    assert missing_dpd_bucket_count == 0, (
        "Missing DPD Bucket values were created."
    )

    print("=" * 80)
    print("DPD DELINQUENCY BUCKET ASSIGNMENT COMPLETED")
    print("=" * 80)

    print(
        "Bucketed Records:",
        dpd_bucket_total
    )
    print(
        "Unexpected DPD Bucket Values:",
        invalid_dpd_bucket_values
    )
    print(
        "Missing DPD Buckets:",
        missing_dpd_bucket_count
    )
    print(
        "DPD Bucket Reconciliation: PASS"
    )

    display(
        dpd_bucket_distribution
    )

    print("\nSample DPD-Bucketed Records:")

    display(
        segmented_portfolio[
            [
                "Customer_ID",
                "Loan_ID",
                "Current_DPD",
                "DPD_Bucket",
            ]
        ]
        .head(15)
    )

    logger.info(
        "DPD Bucket assignment completed "
        "for %s records.",
        dpd_bucket_total
    )

    block_status = "SUCCESS"

except Exception:
    block_status = "FAILED"

    logger.exception(
        "Block 4 failed during DPD Bucket assignment."
    )

    raise

finally:
    block_end_time = datetime.now()

    block_duration_seconds = round(
        (
            block_end_time
            - block_start_time
        ).total_seconds(),
        3
    )

    print("=" * 80)
    print("BLOCK STATUS :", block_status)
    print(
        "END TIME     :",
        block_end_time.strftime(
            "%Y-%m-%d %H:%M:%S"
        )
    )
    print(
        "DURATION (SECONDS):",
        block_duration_seconds
    )
    print("=" * 80)

MODULE 03: RISK SEGMENTATION
BLOCK 4 : ASSIGN DPD DELINQUENCY BUCKET
START TIME : 2026-10-13 09:38:28
DPD DELINQUENCY BUCKET ASSIGNMENT COMPLETED
Bucketed Records: 10000
Unexpected DPD Bucket Values: 0
Missing DPD Buckets: 0
DPD Bucket Reconciliation: PASS


,DPD_Bucket,Record_Count,Portfolio_Percentage
0,Current,8517,85.17
1,1-30,597,5.97
2,31-60,458,4.58
3,61-90,239,2.39
4,90+,189,1.89
5,Unresolved,0,0.00
6,Invalid DPD,0,0.00



Sample DPD-Bucketed Records:


,Customer_ID,Loan_ID,Current_DPD,DPD_Bucket
0,1000001,50000001,0.0,Current
1,1000002,50000002,0.0,Current
2,1000003,50000003,0.0,Current
3,1000004,50000004,0.0,Current
4,1000005,50000005,0.0,Current
5,1000006,50000006,0.0,Current
6,1000007,50000007,0.0,Current
7,1000008,50000008,0.0,Current
8,1000009,50000009,0.0,Current
9,1000010,50000010,0.0,Current


2026-10-13 09:38:28,529 | INFO | DPD Bucket assignment completed for 10000 records.


INFO:03_Risk_Segmentation:DPD Bucket assignment completed for 10000 records.


BLOCK STATUS : SUCCESS
END TIME     : 2026-10-13 09:38:28
DURATION (SECONDS): 0.031


In [6]:
# =============================================================================
# BLOCK 5: SORT AND CREATE RISK-DPD CROSS-TAB
# =============================================================================

block_start_time = datetime.now()
block_status = "STARTED"

print("=" * 80)
print("MODULE 03: RISK SEGMENTATION")
print("BLOCK 5 : SORT AND CREATE RISK-DPD CROSS-TAB")
print(
    "START TIME :",
    block_start_time.strftime(
        "%Y-%m-%d %H:%M:%S"
    )
)
print("=" * 80)

try:
    segmented_portfolio = (
        segmented_portfolio
        .sort_values(
            by=[
                "Customer_ID",
                "Loan_ID",
            ],
            ascending=[
                True,
                True,
            ],
            kind="stable"
        )
        .reset_index(
            drop=True
        )
    )

    sorting_validation = bool(
        segmented_portfolio[
            [
                "Customer_ID",
                "Loan_ID",
            ]
        ]
        .equals(
            segmented_portfolio[
                [
                    "Customer_ID",
                    "Loan_ID",
                ]
            ]
            .sort_values(
                by=[
                    "Customer_ID",
                    "Loan_ID",
                ],
                kind="stable"
            )
            .reset_index(
                drop=True
            )
        )
    )

    risk_dpd_cross_tab = pd.crosstab(
        segmented_portfolio[
            "Risk_Segment"
        ],
        segmented_portfolio[
            "DPD_Bucket"
        ],
        margins=True,
        margins_name="Total"
    )

    cross_tab_total = int(
        risk_dpd_cross_tab.loc[
            "Total",
            "Total"
        ]
    )

    assert sorting_validation, (
        "Final portfolio sorting validation failed."
    )

    assert cross_tab_total == input_record_count, (
        "Risk-DPD cross-tab does not reconcile "
        "to the input record count."
    )

    print("=" * 80)
    print("SORT AND CROSS-TAB VALIDATION COMPLETED")
    print("=" * 80)

    print(
        "Customer_ID and Loan_ID Sort Status:",
        "PASS"
    )
    print(
        "Cross-Tab Total:",
        cross_tab_total
    )
    print(
        "Cross-Tab Reconciliation:",
        "PASS"
    )

    display(
        risk_dpd_cross_tab
    )

    logger.info(
        "Portfolio sorting and Risk-DPD cross-tab "
        "validation completed successfully."
    )

    block_status = "SUCCESS"

except Exception:
    block_status = "FAILED"

    logger.exception(
        "Block 5 failed during sorting or cross-tab validation."
    )

    raise

finally:
    block_end_time = datetime.now()

    block_duration_seconds = round(
        (
            block_end_time
            - block_start_time
        ).total_seconds(),
        3
    )

    print("=" * 80)
    print("BLOCK STATUS :", block_status)
    print(
        "END TIME     :",
        block_end_time.strftime(
            "%Y-%m-%d %H:%M:%S"
        )
    )
    print(
        "DURATION (SECONDS):",
        block_duration_seconds
    )
    print("=" * 80)

MODULE 03: RISK SEGMENTATION
BLOCK 5 : SORT AND CREATE RISK-DPD CROSS-TAB
START TIME : 2026-10-13 09:40:58
SORT AND CROSS-TAB VALIDATION COMPLETED
Customer_ID and Loan_ID Sort Status: PASS
Cross-Tab Total: 10000
Cross-Tab Reconciliation: PASS


DPD_Bucket,1-30,31-60,61-90,90+,Current,Total
Risk_Segment,,,,,,
High Risk,305,235,127,95,1723,2485
Low Risk,39,19,13,15,2449,2535
Medium Risk,253,204,99,79,4345,4980
Total,597,458,239,189,8517,10000


2026-10-13 09:40:58,828 | INFO | Portfolio sorting and Risk-DPD cross-tab validation completed successfully.


INFO:03_Risk_Segmentation:Portfolio sorting and Risk-DPD cross-tab validation completed successfully.


BLOCK STATUS : SUCCESS
END TIME     : 2026-10-13 09:40:58
DURATION (SECONDS): 0.058


In [7]:
# =============================================================================
# BLOCK 6: FINAL VALIDATION AND MODULE SUMMARY
# =============================================================================

block_start_time = datetime.now()
block_status = "STARTED"

print("=" * 80)
print("MODULE 03: RISK SEGMENTATION")
print("BLOCK 6 : FINAL VALIDATION AND MODULE SUMMARY")
print(
    "START TIME :",
    block_start_time.strftime(
        "%Y-%m-%d %H:%M:%S"
    )
)
print("=" * 80)

try:
    output_record_count = len(
        segmented_portfolio
    )

    output_column_count = len(
        segmented_portfolio.columns
    )

    duplicate_customer_ids_after = int(
        segmented_portfolio[
            "Customer_ID"
        ]
        .duplicated()
        .sum()
    )

    duplicate_loan_ids_after = int(
        segmented_portfolio[
            "Loan_ID"
        ]
        .duplicated()
        .sum()
    )

    missing_risk_segments_after = int(
        segmented_portfolio[
            "Risk_Segment"
        ]
        .isna()
        .sum()
    )

    missing_dpd_buckets_after = int(
        segmented_portfolio[
            "DPD_Bucket"
        ]
        .isna()
        .sum()
    )

    final_unclassified_count = int(
        (
            segmented_portfolio[
                "Risk_Segment"
            ] == "Unclassified"
        )
        .sum()
    )

    final_unresolved_dpd_count = int(
        (
            segmented_portfolio[
                "DPD_Bucket"
            ] == "Unresolved"
        )
        .sum()
    )

    final_invalid_dpd_count = int(
        (
            segmented_portfolio[
                "DPD_Bucket"
            ] == "Invalid DPD"
        )
        .sum()
    )

    validation_results = pd.DataFrame({
        "Validation_Check": [
            "Input and Output Record Count",
            "Missing Risk_Segment",
            "Missing DPD_Bucket",
            "Duplicate Customer_ID",
            "Duplicate Loan_ID",
            "Risk Segment Distribution Total",
            "DPD Bucket Distribution Total",
            "Risk-DPD Cross-Tab Total",
            "Customer_ID and Loan_ID Sort",
        ],
        "Expected_Result": [
            input_record_count,
            0,
            0,
            0,
            0,
            input_record_count,
            input_record_count,
            input_record_count,
            "PASS",
        ],
        "Actual_Result": [
            output_record_count,
            missing_risk_segments_after,
            missing_dpd_buckets_after,
            duplicate_customer_ids_after,
            duplicate_loan_ids_after,
            risk_segment_total,
            dpd_bucket_total,
            cross_tab_total,
            (
                "PASS"
                if sorting_validation
                else "FAIL"
            ),
        ],
    })

    validation_results[
        "Status"
    ] = np.where(
        validation_results[
            "Expected_Result"
        ].astype(str)
        ==
        validation_results[
            "Actual_Result"
        ].astype(str),
        "PASS",
        "FAIL"
    )

    module_summary = pd.DataFrame({
        "Metric": [
            "Input Records",
            "Output Records",
            "Input Columns",
            "Output Columns",
            "Low Risk Records",
            "Medium Risk Records",
            "High Risk Records",
            "Unclassified Records",
            "Current DPD Records",
            "1-30 DPD Records",
            "31-60 DPD Records",
            "61-90 DPD Records",
            "90+ DPD Records",
            "Unresolved DPD Records",
            "Invalid DPD Records",
            "Processing Timestamp",
        ],
        "Result": [
            input_record_count,
            output_record_count,
            input_column_count,
            output_column_count,
            int(
                (
                    segmented_portfolio[
                        "Risk_Segment"
                    ] == "Low Risk"
                ).sum()
            ),
            int(
                (
                    segmented_portfolio[
                        "Risk_Segment"
                    ] == "Medium Risk"
                ).sum()
            ),
            int(
                (
                    segmented_portfolio[
                        "Risk_Segment"
                    ] == "High Risk"
                ).sum()
            ),
            final_unclassified_count,
            int(
                (
                    segmented_portfolio[
                        "DPD_Bucket"
                    ] == "Current"
                ).sum()
            ),
            int(
                (
                    segmented_portfolio[
                        "DPD_Bucket"
                    ] == "1-30"
                ).sum()
            ),
            int(
                (
                    segmented_portfolio[
                        "DPD_Bucket"
                    ] == "31-60"
                ).sum()
            ),
            int(
                (
                    segmented_portfolio[
                        "DPD_Bucket"
                    ] == "61-90"
                ).sum()
            ),
            int(
                (
                    segmented_portfolio[
                        "DPD_Bucket"
                    ] == "90+"
                ).sum()
            ),
            final_unresolved_dpd_count,
            final_invalid_dpd_count,
            datetime.now(),
        ],
    })

    print("=" * 80)
    print("FINAL RISK SEGMENTATION VALIDATION")
    print("=" * 80)

    display(
        validation_results
    )

    print("\nRisk Segmentation Module Summary:")

    display(
        module_summary
    )

    failed_checks = validation_results.loc[
        validation_results[
            "Status"
        ] != "PASS"
    ]

    if not failed_checks.empty:
        raise ValueError(
            "Module 03 validation failed. "
            "Review the failed validation checks."
        )

    print(
        "Overall Module 03 Validation Status: PASS"
    )

    logger.info(
        "All Module 03 validation checks passed."
    )

    block_status = "SUCCESS"

except Exception:
    block_status = "FAILED"

    logger.exception(
        "Block 6 failed during final validation."
    )

    raise

finally:
    block_end_time = datetime.now()

    block_duration_seconds = round(
        (
            block_end_time
            - block_start_time
        ).total_seconds(),
        3
    )

    print("=" * 80)
    print("BLOCK STATUS :", block_status)
    print(
        "END TIME     :",
        block_end_time.strftime(
            "%Y-%m-%d %H:%M:%S"
        )
    )
    print(
        "DURATION (SECONDS):",
        block_duration_seconds
    )
    print("=" * 80)

MODULE 03: RISK SEGMENTATION
BLOCK 6 : FINAL VALIDATION AND MODULE SUMMARY
START TIME : 2026-10-13 09:42:19
FINAL RISK SEGMENTATION VALIDATION


,Validation_Check,Expected_Result,Actual_Result,Status
0,Input and Output Record Count,10000,10000,PASS
1,Missing Risk_Segment,0,0,PASS
2,Missing DPD_Bucket,0,0,PASS
3,Duplicate Customer_ID,0,0,PASS
4,Duplicate Loan_ID,0,0,PASS
5,Risk Segment Distribution Total,10000,10000,PASS
6,DPD Bucket Distribution Total,10000,10000,PASS
7,Risk-DPD Cross-Tab Total,10000,10000,PASS
8,Customer_ID and Loan_ID Sort,PASS,PASS,PASS



Risk Segmentation Module Summary:


,Metric,Result
0,Input Records,10000
1,Output Records,10000
2,Input Columns,22
3,Output Columns,24
4,Low Risk Records,2535
5,Medium Risk Records,4980
6,High Risk Records,2485
7,Unclassified Records,0
8,Current DPD Records,8517
9,1-30 DPD Records,597


Overall Module 03 Validation Status: PASS
2026-10-13 09:42:19,183 | INFO | All Module 03 validation checks passed.


INFO:03_Risk_Segmentation:All Module 03 validation checks passed.


BLOCK STATUS : SUCCESS
END TIME     : 2026-10-13 09:42:19
DURATION (SECONDS): 0.077


In [8]:
# =============================================================================
# BLOCK 7: SAVE OUTPUTS, VERIFY FILES, AND CLOSE MODULE
# =============================================================================

block_start_time = datetime.now()
block_status = "STARTED"

print("=" * 80)
print("MODULE 03: RISK SEGMENTATION")
print("BLOCK 7 : SAVE OUTPUTS AND VERIFY FILES")
print(
    "START TIME :",
    block_start_time.strftime(
        "%Y-%m-%d %H:%M:%S"
    )
)
print("=" * 80)

try:
    segmented_portfolio.to_csv(
        OUTPUT_FILE,
        index=False,
        date_format="%Y-%m-%d"
    )

    module_summary.to_csv(
        SUMMARY_FILE,
        index=False
    )

    validation_results.to_csv(
        VALIDATION_FILE,
        index=False
    )

    risk_segment_distribution.to_csv(
        RISK_DISTRIBUTION_FILE,
        index=False
    )

    dpd_bucket_distribution.to_csv(
        DPD_DISTRIBUTION_FILE,
        index=False
    )

    risk_dpd_cross_tab.to_csv(
        CROSS_TAB_FILE
    )

    logger.info(
        "Segmented portfolio and validation evidence "
        "saved successfully."
    )

    expected_files = [
        OUTPUT_FILE,
        SUMMARY_FILE,
        VALIDATION_FILE,
        RISK_DISTRIBUTION_FILE,
        DPD_DISTRIBUTION_FILE,
        CROSS_TAB_FILE,
        LOG_FILE,
    ]

    output_verification = pd.DataFrame({
        "File_Name": [
            file_path.name
            for file_path in expected_files
        ],
        "Exists": [
            file_path.exists()
            for file_path in expected_files
        ],
        "Size_Bytes": [
            (
                file_path.stat().st_size
                if file_path.exists()
                else 0
            )
            for file_path in expected_files
        ],
        "Full_Path": [
            str(file_path)
            for file_path in expected_files
        ],
    })

    if not output_verification[
        "Exists"
    ].all():
        raise FileNotFoundError(
            "One or more Module 03 output files "
            "were not created."
        )

    block_status = "SUCCESS"

except Exception:
    block_status = "FAILED"

    logger.exception(
        "Block 7 failed while saving or verifying outputs."
    )

    raise

finally:
    block_end_time = datetime.now()

    block_duration_seconds = round(
        (
            block_end_time
            - block_start_time
        ).total_seconds(),
        3
    )

    print("=" * 80)
    print("BLOCK STATUS :", block_status)
    print(
        "END TIME     :",
        block_end_time.strftime(
            "%Y-%m-%d %H:%M:%S"
        )
    )
    print(
        "DURATION (SECONDS):",
        block_duration_seconds
    )
    print("=" * 80)

module_end_timestamp = datetime.now()

module_duration_seconds = round(
    (
        module_end_timestamp
        - module_start_timestamp
    ).total_seconds(),
    3
)

print("=" * 80)
print("PYTHON V2 - MODULE 03 COMPLETED")
print("RISK SEGMENTATION EXECUTED SUCCESSFULLY")
print("=" * 80)

print("Input Records:", input_record_count)
print("Output Records:", output_record_count)
print("Segmented Output:", OUTPUT_FILE)
print("Validation Summary:", SUMMARY_FILE)
print("Validation Evidence:", VALIDATION_FILE)
print("Risk Distribution:", RISK_DISTRIBUTION_FILE)
print("DPD Distribution:", DPD_DISTRIBUTION_FILE)
print("Risk-DPD Cross-Tab:", CROSS_TAB_FILE)
print("Execution Log:", LOG_FILE)

print("\nOutput File Verification:")

display(
    output_verification
)

print("=" * 80)
print("MODULE STATUS : SUCCESS")
print(
    "MODULE START :",
    module_start_timestamp.strftime(
        "%Y-%m-%d %H:%M:%S"
    )
)
print(
    "MODULE END   :",
    module_end_timestamp.strftime(
        "%Y-%m-%d %H:%M:%S"
    )
)
print(
    "TOTAL DURATION (SECONDS):",
    module_duration_seconds
)
print("=" * 80)

print(
    "All Module 03 output files verified successfully."
)

logger.info(
    "Module 03 completed successfully in %s seconds.",
    module_duration_seconds
)

MODULE 03: RISK SEGMENTATION
BLOCK 7 : SAVE OUTPUTS AND VERIFY FILES
START TIME : 2026-10-13 09:44:44
2026-10-13 09:44:44,923 | INFO | Segmented portfolio and validation evidence saved successfully.


INFO:03_Risk_Segmentation:Segmented portfolio and validation evidence saved successfully.


BLOCK STATUS : SUCCESS
END TIME     : 2026-10-13 09:44:44
DURATION (SECONDS): 0.335
PYTHON V2 - MODULE 03 COMPLETED
RISK SEGMENTATION EXECUTED SUCCESSFULLY
Input Records: 10000
Output Records: 10000
Segmented Output: /content/drive/MyDrive/Banking_Stress_Testing_V2_Python/output/segmented_portfolio.csv
Validation Summary: /content/drive/MyDrive/Banking_Stress_Testing_V2_Python/validation/03_risk_segmentation_summary.csv
Validation Evidence: /content/drive/MyDrive/Banking_Stress_Testing_V2_Python/validation/03_risk_segmentation_validation.csv
Risk Distribution: /content/drive/MyDrive/Banking_Stress_Testing_V2_Python/validation/03_risk_segment_distribution.csv
DPD Distribution: /content/drive/MyDrive/Banking_Stress_Testing_V2_Python/validation/03_dpd_bucket_distribution.csv
Risk-DPD Cross-Tab: /content/drive/MyDrive/Banking_Stress_Testing_V2_Python/validation/03_risk_dpd_cross_tab.csv
Execution Log: /content/drive/MyDrive/Banking_Stress_Testing_V2_Python/logs/03_Risk_Segmentation.log

Ou

,File_Name,Exists,Size_Bytes,Full_Path
0,segmented_portfolio.csv,True,1764722,/content/drive/MyDrive/Banking_Stress_Testing_...
1,03_risk_segmentation_summary.csv,True,388,/content/drive/MyDrive/Banking_Stress_Testing_...
2,03_risk_segmentation_validation.csv,True,399,/content/drive/MyDrive/Banking_Stress_Testing_...
3,03_risk_segment_distribution.csv,True,129,/content/drive/MyDrive/Banking_Stress_Testing_...
4,03_dpd_bucket_distribution.csv,True,156,/content/drive/MyDrive/Banking_Stress_Testing_...
5,03_risk_dpd_cross_tab.csv,True,183,/content/drive/MyDrive/Banking_Stress_Testing_...
6,03_Risk_Segmentation.log,True,644,/content/drive/MyDrive/Banking_Stress_Testing_...


MODULE STATUS : SUCCESS
MODULE START : 2026-10-13 09:27:39
MODULE END   : 2026-10-13 09:44:44
TOTAL DURATION (SECONDS): 1025.246
All Module 03 output files verified successfully.
2026-10-13 09:44:44,944 | INFO | Module 03 completed successfully in 1025.246 seconds.


INFO:03_Risk_Segmentation:Module 03 completed successfully in 1025.246 seconds.
